# Sneha Singh
## DATA 266 — Homework 1 (CUDA)
Matrix multiplication with blocks and threads, timing vs CPU, profiler

Run this in **Google Colab** with a GPU: Runtime → Change runtime type → Hardware accelerator → GPU.


In [1]:
SID4 = 670
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

print("SID4:", SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)


SID4: 670
SEED: 670
SLICE: 670
HP_ID: 4
CLS_A: 0
CLS_B: 5


In [2]:
# should say Tesla / T4 / A100 / L4 etc
!nvidia-smi


Mon Aug 31 03:15:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Blocks and threads

I am multiplying two N x N matrices, C = A * B.

The GPU kernel uses a **2D grid of blocks**. Each block is **16 x 16 threads**.

- `threadIdx.x` / `threadIdx.y` = which thread I am inside my block (0 to 15)
- `blockIdx.x` / `blockIdx.y` = which block I am in the grid
- `blockDim.x` / `blockDim.y` = 16 (threads per block in each direction)

So the output index is:

```
row = blockIdx.y * blockDim.y + threadIdx.y
col = blockIdx.x * blockDim.x + threadIdx.x
```

That thread computes **one** value `C[row, col]`. Number of blocks is `ceil(N/16)` in x and y so the grid covers the whole matrix. If N is not a multiple of 16, extra threads hit `if (row < N && col < N)` and do nothing.


In [3]:
%%writefile matmul.cu
/*
 * DATA 266 HW1 — C = A * B (square matrices)
 * SID4 = 0670
 *
 * How blocks and threads work here:
 *   I launch a 2D grid of blocks. Each block is 16 x 16 threads.
 *   thread (threadIdx.x, threadIdx.y) inside block (blockIdx.x, blockIdx.y)
 *   computes ONE output:
 *      row = blockIdx.y * 16 + threadIdx.y
 *      col = blockIdx.x * 16 + threadIdx.x
 *      C[row, col] = sum_k A[row, k] * B[k, col]
 *   Extra threads (when N is not a multiple of 16) just return.
 */

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>
#include <cuda_runtime.h>

#define THREADS 16

void cpu_matmul(const float *A, const float *B, float *C, int N) {
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            float s = 0.0f;
            for (int k = 0; k < N; k++) {
                s += A[i * N + k] * B[k * N + j];
            }
            C[i * N + j] = s;
        }
    }
}

__global__ void gpu_matmul(const float *A, const float *B, float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float s = 0.0f;
        for (int k = 0; k < N; k++) {
            s += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = s;
    }
}

double ms_now(struct timespec t0, struct timespec t1) {
    return (t1.tv_sec - t0.tv_sec) * 1000.0 + (t1.tv_nsec - t0.tv_nsec) / 1e6;
}

int main(void) {
    int sizes[3] = {256, 1024, 4096};
    printf("SID4=0670  C = A * B\n");
    printf("each block is %d x %d threads\n", THREADS, THREADS);
    printf("each thread writes one C[row,col]\n\n");
    printf("%8s %12s %14s %14s %10s\n", "N", "CPU_ms", "kernel_ms", "H2D+D2H_ms", "speedup");

    for (int t = 0; t < 3; t++) {
        int N = sizes[t];
        size_t bytes = (size_t)N * (size_t)N * sizeof(float);

        float *A = (float *)malloc(bytes);
        float *B = (float *)malloc(bytes);
        float *Ccpu = (float *)malloc(bytes);
        float *Cgpu = (float *)malloc(bytes);
        for (int i = 0; i < N * N; i++) {
            A[i] = (i % 13) * 0.01f;
            B[i] = (i % 7) * 0.02f;
        }

        // CPU baseline
        struct timespec t0, t1;
        clock_gettime(CLOCK_MONOTONIC, &t0);
        cpu_matmul(A, B, Ccpu, N);
        clock_gettime(CLOCK_MONOTONIC, &t1);
        double cpu_ms = ms_now(t0, t1);

        float *dA, *dB, *dC;
        cudaMalloc((void **)&dA, bytes);
        cudaMalloc((void **)&dB, bytes);
        cudaMalloc((void **)&dC, bytes);

        dim3 threads(THREADS, THREADS);
        dim3 blocks((N + THREADS - 1) / THREADS, (N + THREADS - 1) / THREADS);

        // warmup so the timed run is not the first launch
        cudaMemcpy(dA, A, bytes, cudaMemcpyHostToDevice);
        cudaMemcpy(dB, B, bytes, cudaMemcpyHostToDevice);
        gpu_matmul<<<blocks, threads>>>(dA, dB, dC, N);
        cudaDeviceSynchronize();

        cudaEvent_t e0, e1, e2, e3;
        cudaEventCreate(&e0);
        cudaEventCreate(&e1);
        cudaEventCreate(&e2);
        cudaEventCreate(&e3);

        cudaEventRecord(e0);
        cudaMemcpy(dA, A, bytes, cudaMemcpyHostToDevice);
        cudaMemcpy(dB, B, bytes, cudaMemcpyHostToDevice);
        cudaEventRecord(e1);
        gpu_matmul<<<blocks, threads>>>(dA, dB, dC, N);
        cudaEventRecord(e2);
        cudaMemcpy(Cgpu, dC, bytes, cudaMemcpyDeviceToHost);
        cudaEventRecord(e3);
        cudaEventSynchronize(e3);

        float h2d = 0, kern = 0, d2h = 0;
        cudaEventElapsedTime(&h2d, e0, e1);
        cudaEventElapsedTime(&kern, e1, e2);
        cudaEventElapsedTime(&d2h, e2, e3);
        float xfer = h2d + d2h;
        float e2e = h2d + kern + d2h;
        float speedup = (float)(cpu_ms / e2e);

        printf("%8d %12.3f %14.3f %14.3f %10.3f\n", N, cpu_ms, kern, xfer, speedup);

        if (N == 256) {
            float maxdiff = 0.0f;
            for (int i = 0; i < N * N; i++) {
                float d = fabsf(Ccpu[i] - Cgpu[i]);
                if (d > maxdiff) maxdiff = d;
            }
            printf("  N=256 max |CPU-GPU| = %g\n", maxdiff);
        }

        cudaFree(dA);
        cudaFree(dB);
        cudaFree(dC);
        free(A);
        free(B);
        free(Ccpu);
        free(Cgpu);
        cudaEventDestroy(e0);
        cudaEventDestroy(e1);
        cudaEventDestroy(e2);
        cudaEventDestroy(e3);
    }
    return 0;
}


Writing matmul.cu


In [4]:
# compile and run (N=256 will also check CPU vs GPU)
!nvcc -O3 matmul.cu -o matmul
!./matmul


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
SID4=0670  C = A * B
each block is 16 x 16 threads
each thread writes one C[row,col]

       N       CPU_ms      kernel_ms     H2D+D2H_ms    speedup
     256       25.308          0.146          0.437     43.376
  N=256 max |CPU-GPU| = 1.78814e-07
    1024     3291.187          9.187          4.756    236.054
    4096   763322.870        362.831         72.603   1753.016


### Timing table

From the `./matmul` run on Colab (Tesla T4). Speedup = CPU / (kernel + H2D+D2H).

| Matrix size | CPU (ms) | GPU kernel (ms) | H2D+D2H (ms) | Speedup |
|-------------|----------|-----------------|--------------|---------|
| 256 | 25.308 | 0.146 | 0.437 | 43.38 |
| 1024 | 3291.187 | 9.187 | 4.756 | 236.05 |
| 4096 | 763322.870 | 362.831 | 72.603 | 1753.02 |

N=256 check: max |CPU-GPU| = 1.79e-07.

Profiler: **nvprof** (nsys was not installed). GPU activities: `gpu_matmul` vs `[CUDA memcpy HtoD]` vs `[CUDA memcpy DtoH]`.


In [5]:
# profiler — try nsys first, then nvprof
!nsys profile --stats=true --force-overwrite=true -o nsys_matmul ./matmul


/bin/bash: line 1: nsys: command not found


In [6]:
# nsys was not installed on this Colab, try nvprof
!nvprof ./matmul


SID4=0670  C = A * B
each block is 16 x 16 threads
each thread writes one C[row,col]

       N       CPU_ms      kernel_ms     H2D+D2H_ms    speedup
==4882== NVPROF is profiling process 4882, command: ./matmul
     256       20.235          0.073          0.395     43.276
  N=256 max |CPU-GPU| = 1.78814e-07
    1024     3384.626          9.202          5.445    231.077
    4096   703483.422        364.315         72.518   1610.418
==4882== Profiling application: ./matmul
==4882== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   88.84%  849.37ms         6  141.56ms  59.358us  466.57ms  gpu_matmul(float const *, float const *, float*, int)
                    6.57%  62.813ms        12  5.2344ms  23.359us  16.882ms  [CUDA memcpy HtoD]
                    4.59%  43.907ms         3  14.636ms  21.120us  42.043ms  [CUDA memcpy DtoH]
      API calls:   44.29%  485.45ms        15  32.364ms  66.961us  407.65ms  cudaMemcpy
    

If `nsys` is missing, run this cell instead:

```
!nvprof ./matmul
```

Paste the profiler lines that show **kernel** vs **CUDA memcpy HtoD / DtoH**. That is what the homework wants (kernel time separate from transfer time).


### Crossover

The smallest size in my table where GPU end-to-end is faster than CPU is **N = 256** (about 43x). I did not measure anything smaller than 256.

It is not size 0 because A and B still have to be copied to the GPU and C copied back. At N=256 those copies are 0.44 ms and the kernel is only 0.15 ms, so transfer is most of the GPU time. CPU is still 25 ms, so GPU wins anyway. nvprof shows the same split: memcpy HtoD + DtoH vs the `gpu_matmul` kernel. At 1024 and 4096 the N^3 work grows faster than the N^2 copies, so speedup goes to about 236x and 1753x.
